<a href="https://colab.research.google.com/github/marianaolmedo/Simulacion-II/blob/main/L%C3%ADnea_de_espera_con_dos_servidores_en_serie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Línea de espera con dos servidores en serie**



# **Taxis en el Aeropuerto con Dos Etapas de Servicio en Serie**

Modelo: Llegadas → Servidor 1 → Servidor 2 → Salida

Colas y servicios: Exponenciales (M/M/1 en cada etapa).

*	Servidor 1: Punto de verificación, donde cada taxi pasa un control de seguridad rápido.

* Servidor 2: Estación de asignación de pasajero, donde se confirma destino y se emite la salida.

* Toda la fila: Los taxis esperan primero para el control, luego para la asignación del cliente.

**Datos del Ejemplo:**

$λ = 12$ taxis por hora (uno cada 5 minutos en promedio)

**Etapa 1: Control de acceso (S1)**

Antes de entrar a la zona de pasajeros, el taxi debe pasar por un control rápido (revisión visual y registro).

* Duración del servicio: exponencial promedio = 2 minutos

  $\mu_1 = 0.5 \text{ servicios/minuto}$

**Etapa 2: Asignación del pasajero (S2)**

Una vez aprobado, el taxi pasa al área donde se le asigna un pasajero concreto y se coordinan destino y tarifa.

* Duración del servicio: exponencial promedio = 3 minutos

   $\mu_2 = \frac{1}{3} \text{ servicios/minuto}$

**Parametros:**

Se tiene convertir todo a minutos

* Llegadas:

  $λ = \frac{12}{60} = 0.2 \text{ taxis/minuto}$

* Salidas:

  $\mu_1 = 0.5 $

  $\mu_2 = \frac{1}{3}$

**Pasos para construir la Simulación:**

1. Poner las librerias necesarias para hacer funcionar el código:

In [58]:
import numpy as np
from random import expovariate
from statistics import mean
from math import inf
import matplotlib.pyplot as plt

2. Parámetros y variables del modelo

In [59]:
lamb = 0.2     # tasa de llegada
mu1 = 0.5        # tasa servicio servidor 1
mu2 = 1/3        # tasa servicio servidor 2
Num_PK = 5000    # Número de clientes a simular

t = 0.0
n1 = 0          # número de clientes en S1 (fila + servicio)
n2 = 0          # número de clientes en S2
NLL = 0         # número de llegadas realizadas
NS = 0          # clientes que ya salieron


t_LL = np.random.exponential(1/lamb)   # próxima llegada
t1 = np.inf     # próxima salida de S1
t2 = np.inf     # próxima salida de S2

LL1 = []   # tiempo llegada a S1
LL2 = []   # tiempo llegada a S2
S = []     # tiempo salida del sistema

3. Simulación:

In [60]:
while NS < Num_PK:

    if t_LL <= t1 and t_LL <= t2:
        # Caso 1: llegada
        t = t_LL
        NLL = NLL + 1
        n1 = n1 + 1
        LL1.append(t)

        t_LL = t + np.random.exponential(1/lamb)

        if n1 == 1:
            Y1 = np.random.exponential(1/mu1)
            t1 = t + Y1

    elif t1 < t_LL and t1 <= t2:
        # Caso 2: salida de  S1 → entrada a S2
        t = t1
        n1 = n1 - 1
        n2 = n2 + 1
        LL2.append(t)

        if n1 == 0:
            t1 = np.inf
        else:
            Y1 = np.random.exponential(1/mu1)
            t1 = t + Y1


        if n2 == 1:
            Y2 = np.random.exponential(1/mu2)
            t2 = t + Y2

    else:
        # Caso 3: salida de S2
        t = t2
        NS += 1
        n2 -= 1
        S.append(t)

        if n2 > 0:
            Y2 = np.random.exponential(1/mu2)
            t2 = t + Y2
        else:
            t2 = np.inf


4. Resultados:

In [61]:
LL1 = np.array(LL1)
LL2 = np.array(LL2)
S = np.array(S)

#  Tiempo promedio en el Sistema:
W = np.mean(S - LL1[:NS])

# Tiempo proemdio en filas:
Wq1 = np.mean((LL2[:NS] - LL1[:NS]))
Wq2 = np.mean((S - LL2[:NS])) - (1/mu2)
Wq = Wq1 + Wq2

print(f"\nTiempo promedio en el Sistema (Por Simulación): \n W  = {W:.4f}")
print(f"\nTiempo promedio en Filas (Por Simulación): \n Wq  = {Wq:.4f}")
print(f"   - Espera en S1 (Wq1): {Wq1:.4f}")
print(f"   - Espera en S2 (Wq2): {Wq2:.4f}")



Tiempo promedio en el Sistema (Por Simulación): 
 W  = 10.7217

Tiempo promedio en Filas (Por Simulación): 
 Wq  = 7.7217
   - Espera en S1 (Wq1): 3.5195
   - Espera en S2 (Wq2): 4.2023


**Fórmulas analíticas del sistema en serie**

Cada etapa funciona como un M/M/1 independiente, con la misma tasa de llegada $λ$ porque todos los taxis pasan por ambos.

**Etapa 1: Control de acceso (S1):**

In [62]:
rho_1 = lamb/mu1
print(f"rho_1  = {rho_1:.4f}")

rho_1  = 0.4000


1. Tiempo de espera en la fila

$$Wq_1= \frac{\rho^2}{λ(1-\rho_1)}=\frac{(0.4)^2}{0.2(1-0.4)}=1.333\text{ min}$$

In [63]:
Wq1_teo = (rho_1)**2/(0.2*(1-rho_1))
print(f"Wq1  = {Wq1_teo:.4f}")

Wq1  = 1.3333


2. Tiempo total en S1 (espera + servicio)
$$ W_1 = \frac{1}{\mu_1 - λ} = \frac{1}{0.5 - 0.2} = 3.333 \text{ min}$$

In [64]:
W1 = 1/(mu1 - lamb)
print(f"W1  = {W1:.4f}")

W1  = 3.3333


**Etapa 2: Asignación del pasajero (S2):**

In [65]:
rho_2 = lamb/mu2
print(f"rho_2  = {rho_2:.4f}")

rho_2  = 0.6000


1. Tiempo de espera en la fila

$$Wq_2= \frac{\rho^2}{λ(1-\rho_2)}=\frac{(0.6)^2}{0.2(1-0.6)}=4.5\text{ min}$$

In [66]:
Wq2_teo = (rho_2)**2/(0.2*(1-rho_2))
print(f"Wq2  = {Wq2_teo:.4f}")

Wq2  = 4.5000


2. Tiempo total en S1 (espera + servicio)
$$ W_2 = \frac{1}{\mu_2 - λ} = \frac{1}{0.333 - 0.2} = 7.5 \text{ min}$$

In [67]:
W2 = 1/(mu2 - lamb)
print(f"W2  = {W2:.4f}")

W2  = 7.5000


**Resultados:**

In [68]:
# Tiempo total en el sistema
W_teo = W1 + W2
print(f"\nTiempo promedio en el Sistema (Por Formulas): \n W  = {W_teo:.4f}")

# Tiempo total en espera
Wq_teo = Wq1_teo + Wq2_teo
print(f"\nTiempo promedio en Filas (Por Formulas): \n Wq  = {Wq_teo:.4f}")
print(f"   - Espera en S1 (Wq1): {Wq1_teo:.4f}")
print(f"   - Espera en S2 (Wq2): {Wq2_teo:.4f}")


Tiempo promedio en el Sistema (Por Formulas): 
 W  = 10.8333

Tiempo promedio en Filas (Por Formulas): 
 Wq  = 5.8333
   - Espera en S1 (Wq1): 1.3333
   - Espera en S2 (Wq2): 4.5000


**Compraración de resultados**

In [70]:
diff_W   = abs(W_teo  - W)
diff_Wq  = abs(Wq_teo - Wq)

print(f"ΔW  = {diff_W:.6f}")
print(f"ΔWq = {diff_Wq:.6f}")


ΔW  = 0.111601
ΔWq = 1.888399
